# GUI prototype — ipywidgets

Wraps a small subset of `parallel_stereo` in ipywidgets. Adjust the controls; the generated CLI updates live; click **Run** to execute (or dry-run).

Exploratory — see `experiments/README.md` for context.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import shlex, subprocess

## Parameters

In [ ]:
algo = widgets.Dropdown(
    options=['asp_bm', 'asp_mgm', 'asp_sgm'],
    value='asp_mgm',
    description='Algorithm:',
)
subpixel = widgets.Dropdown(
    options=[('1 (parabolic)', 1), ('2 (Bayes EM)', 2), ('9 (Bayes EM + MGM)', 9)],
    value=9,
    description='Subpixel:',
)
processes = widgets.IntSlider(value=4, min=1, max=16, description='Processes:')
threads_mp = widgets.IntSlider(value=2, min=1, max=8, description='Threads/proc:')

left_image = widgets.Text(value='left.tif', description='Left img:')
right_image = widgets.Text(value='right.tif', description='Right img:')
left_cam = widgets.Text(value='left.xml', description='Left cam:')
right_cam = widgets.Text(value='right.xml', description='Right cam:')
out_prefix = widgets.Text(value='out_stereo/run', description='Output:')

dry_run = widgets.Checkbox(value=True, description='Dry run (do not execute)')

## Live CLI preview

In [ ]:
cmd_preview = widgets.Output()

def build_cmd():
    return [
        'parallel_stereo',
        '--stereo-algorithm', algo.value,
        '--subpixel-mode', str(subpixel.value),
        '--processes', str(processes.value),
        '--threads-multiprocess', str(threads_mp.value),
        left_image.value, right_image.value,
        left_cam.value, right_cam.value,
        out_prefix.value,
    ]

def update_preview(*_):
    cmd_preview.clear_output()
    with cmd_preview:
        print(shlex.join(build_cmd()))

for w in (algo, subpixel, processes, threads_mp,
          left_image, right_image, left_cam, right_cam, out_prefix):
    w.observe(update_preview, names='value')

## Run

In [ ]:
run_btn = widgets.Button(description='Run', button_style='primary')
run_output = widgets.Output()

def on_run(_):
    cmd = build_cmd()
    run_output.clear_output()
    with run_output:
        if dry_run.value:
            print('DRY RUN — would execute:')
            print(shlex.join(cmd))
            return
        print('$', shlex.join(cmd))
        print('Running... (subprocess is synchronous; UI blocks until it returns)')
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            print(f'returncode: {result.returncode}')
            print('--- stdout ---')
            print(result.stdout if result.stdout else '<empty>')
            print('--- stderr ---')
            print(result.stderr if result.stderr else '<empty>')
            if not result.stdout and not result.stderr:
                print()
                print('Note: ASP tools sometimes write directly to /dev/tty '
                      'and bypass capture_output. Run the same command in a '
                      'terminal to see the real output.')
        except subprocess.TimeoutExpired:
            print('TIMEOUT after 60s — subprocess killed.')
        except FileNotFoundError as e:
            print(f'Binary not on PATH: {e}')
        except Exception:
            import traceback
            print('Unexpected exception:')
            traceback.print_exc()
run_btn.on_click(on_run)

## App

In [ ]:
ui = widgets.VBox([
    widgets.HTML('<h4>Stereo parameters</h4>'),
    algo, subpixel, processes, threads_mp,
    widgets.HTML('<h4>Inputs and output</h4>'),
    left_image, right_image, left_cam, right_cam, out_prefix,
    widgets.HTML('<h4>Generated CLI</h4>'),
    cmd_preview,
    widgets.HTML('<h4>Run</h4>'),
    dry_run, run_btn, run_output,
])
display(ui)
update_preview()

## Notes

- File-path inputs are plain text. A real GUI would use a file picker — see [ipyfilechooser](https://github.com/crahan/ipyfilechooser).
- Only a tiny slice of `parallel_stereo` flags is exposed. The full set is in [the ASP docs](https://stereopipeline.readthedocs.io/en/latest/correlation.html).
- Output streams as one block at the end; long-running commands need a tail/streaming approach (e.g. `subprocess.Popen` + reading lines into `cmd_preview`).
- Compare the same prototype in solara: `experiments/gui_solara.ipynb`.